In [1]:
import boto3
import json
import os
import random
import time
from datetime import datetime, timedelta
from backend.email_utils import EmailHandler
from botocore.exceptions import ClientError

class AddChildHandler:
    def __init__(self, region="ap-south-1"):
        self.dynamodb = boto3.resource(
            "dynamodb",
            aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
            aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
            region_name=region
        )
        self.parents_table = self.dynamodb.Table("Parents")
        self.children_table = self.dynamodb.Table("Students")
        self.email_handler = EmailHandler()

    def add_child_by_email(self, event):
        """Send OTP to child’s email for verification"""
        try:
            body = json.loads(event.get("body", "{}"))
            parent_email = body.get("parent_email")
            child_email = body.get("child_email")

            if not parent_email or not child_email:
                return self._response(400, {"error": "Both parent_email and child_email are required"})

            # Check if child exists in DynamoDB
            child_record = self.children_table.get_item(Key={"email": child_email}).get("Item")
            if not child_record:
                return self._response(404, {"error": "No child found with this email"})

            # Generate OTP
            otp = str(random.randint(100000, 999999))
            expiry = int(time.time()) + 600  # 10 minutes

            # Store OTP in child’s record
            self.children_table.update_item(
                Key={"email": child_email},
                UpdateExpression="SET pending_otp = :otp, otp_expiry = :exp",
                ExpressionAttributeValues={":otp": otp, ":exp": expiry}
            )

            # Send OTP email
            email_sent = self.email_handler.send_otp_email(child_email, otp)
            if not email_sent:
                return self._response(500, {"error": "Failed to send OTP email"})

            return self._response(200, {"message": "OTP sent successfully"})

        except ClientError as e:
            return self._response(400, {"error": str(e)})

    def _response(self, status, body):
        return {
            "statusCode": status,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps(body)
        }


In [2]:
obj=AddChildHandler()


dipdap90@gmail.com
cnuk rgfb loih vyem


In [3]:
test_event = {
    "body": json.dumps({
        "parent_email": "parent@example.com",
        "child_email": "bilalahmed7355@gmail.com"
    })
}

# Run the function locally
response = obj.add_child_by_email(test_event)

OTP sent successfully to bilalahmed7355@gmail.com


In [4]:
response

{'statusCode': 200,
 'headers': {'Content-Type': 'application/json'},
 'body': '{"message": "OTP sent successfully"}'}

In [5]:
import boto3
import json
import os
import time
from datetime import datetime
from botocore.exceptions import ClientError

class VerifyChildOTPHandler:
    def __init__(self, region="ap-south-1"):
        self.dynamodb = boto3.resource(
            "dynamodb",
            aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
            aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
            region_name=region
        )
        self.parents_table = self.dynamodb.Table("Parents")
        self.children_table = self.dynamodb.Table("Students")

    def verify_child_otp(self, event):
        """Verify OTP and link child to parent"""
        try:
            body = json.loads(event.get("body", "{}"))
            parent_email = body.get("parent_email")
            child_email = body.get("child_email")
            otp = body.get("otp")

            if not all([parent_email, child_email, otp]):
                return self._response(400, {"error": "Missing required fields"})

            # Get child record
            child = self.children_table.get_item(Key={"email": child_email}).get("Item")
            if not child:
                return self._response(404, {"error": "Child not found"})

            # Validate OTP
            if str(child.get("pending_otp")) != str(otp):
                return self._response(400, {"error": "Invalid OTP"})

            if time.time() > child.get("otp_expiry", 0):
                return self._response(400, {"error": "OTP expired"})

            # Update records: link child with parent
            self.children_table.update_item(
                Key={"email": child_email},
                UpdateExpression="REMOVE pending_otp, otp_expiry SET parent_email = :p",
                ExpressionAttributeValues={":p": parent_email}
            )

            self.parents_table.update_item(
                Key={"email": parent_email},
                UpdateExpression="SET children = list_append(if_not_exists(children, :empty), :child)",
                ExpressionAttributeValues={":child": [child_email], ":empty": []}
            )

            return self._response(200, {"message": "Child successfully linked to parent"})

        except ClientError as e:
            return self._response(400, {"error": str(e)})

    def _response(self, status, body):
        return {
            "statusCode": status,
            "headers": {"Content-Type": "application/json"},
            "body": json.dumps(body)
        }


In [6]:
obj1=VerifyChildOTPHandler()

In [11]:
test_event = {
    "body": json.dumps({
        "parent_email": "parent@example.com",
        "child_email": "bilalahmed7355@gmail.com",
        "otp":"558060"
    })
}


In [12]:
obj1.verify_child_otp(test_event)

{'statusCode': 400,
 'headers': {'Content-Type': 'application/json'},
 'body': '{"error": "Invalid OTP"}'}